# Task A · Run 21: Character-only SVM as a third member

| | |
|---|---|
| **Status** | completed |
| **Outcome** | Character-only TF-IDF as a third member beside word+char SVM and MuRIL: searched out-of-fold 0.8270 / 0.8271, nested 0.8250 against 0.8184 for the best single model. |
| **Record** | [`results/experiments/task_a/21_char_svm_third_member_oof.json`](../../../results/experiments/task_a/21_char_svm_third_member_oof.json) |
| **Write-up** | [docs/experiments.md](../../../docs/experiments.md#a21) |

**Running it.** Add the organizer CSVs and `offenseval_kn.csv` to the session as a private Kaggle dataset (see `data/README.md`), switch Internet on, and use Save & Run All. The setup cell clones this repository, puts `src/` on the path and runs `python -m hastika.research.data`, which copies the files, checks their checksums and rebuilds the fixed holdouts. Research code lives under `hastika.research`. The code and run numbers match this repository; the notebook has not been re-executed since.

The cells below are the notebook as written for the run. Plans and "current best" statements in them describe that moment; the outcome above is what happened.

# Task A: character-only SVM as a third ensemble member

This notebook tests whether a character-only TF-IDF model adds useful errors to the current word+character SVM and MuRIL ensemble. It uses identical five-fold splits for all three members, searches the three-way blend, and reports the nested estimate. The character-only member uses `char_wb` 2–5 grams, calibrated LinearSVC, balanced class weighting, and demojized text.

This is evaluation-only: it produces OOF probabilities and diagnostic files, not a submission ZIP. Expected runtime is about 3–5 hours on a Kaggle T4 ×2; the SVMs themselves are CPU-only and quick.

In [ ]:
import os, pathlib, shutil, subprocess, sys
WORK = '/kaggle/working/hastika'
if os.path.isdir(WORK + '/.git'):
    subprocess.run(['git', '-C', WORK, 'pull', '--ff-only', 'origin', 'main'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', '-b', 'main', '--depth', '1', 'https://github.com/Aaryan127/ProjectMANAS-HastikaICON2026.git', WORK], check=True)
os.chdir(WORK)
os.environ['PYTHONPATH'] = os.path.join(WORK, 'src')
sys.path.insert(0, os.path.join(WORK, 'src'))
pathlib.Path('artifacts/logs').mkdir(parents=True, exist_ok=True)
subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"', shell=True, check=True)
# Organizer CSVs + offenseval_kn.csv, attached as a private Kaggle dataset (see
# data/README.md): copy, verify checksums, rebuild the fixed holdouts.
subprocess.run([sys.executable, "-m", "hastika.research.data"], check=True)
import numpy as np, pandas as pd, torch
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.svm import LinearSVC
from hastika.research.common.preprocessing import clean, dedupe_index
assert torch.cuda.is_available(), 'enable a GPU for the MuRIL member'

def run(cmd, log=None):
    print('$', ' '.join(map(str, cmd)), flush=True)
    fh = open(log, 'w') if log else None
    p = subprocess.Popen([str(x) for x in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        sys.stdout.write(line)
        if fh: fh.write(line)
    p.wait()
    if fh: fh.close()
    if p.returncode: raise RuntimeError(f'exit {p.returncode}: {cmd}')


## 1. Stage the common five-fold corpus

In [ ]:
from pathlib import Path
combined = pd.concat([pd.read_csv('data/raw/binary_train.csv'), pd.read_csv('data/raw/hastika_binary_validation.csv')], ignore_index=True)
combined = combined.drop_duplicates('id', keep='first').reset_index(drop=True)
assert len(combined) == 7252 and combined['id'].is_unique
combined_path = Path('data/derived/task_a_run21_combined.csv')
combined_path.parent.mkdir(parents=True, exist_ok=True)
combined.to_csv(combined_path, index=False)
CANON_TRAIN = Path('data/raw/binary_train.csv')
BACKUP_TRAIN = Path('data/raw/binary_train.csv.before_run21')
assert not BACKUP_TRAIN.exists(), 'stale backup exists; stop and inspect before rerunning'
shutil.copy2(CANON_TRAIN, BACKUP_TRAIN)
shutil.copy2(combined_path, CANON_TRAIN)
fit_keep = dedupe_index(combined['Comment'].tolist(), combined['Label'].tolist(), 'task A')
fit_df = combined.iloc[fit_keep].reset_index(drop=True)
y = (fit_df['Label'] == 'Hate').astype(int).to_numpy()
X = fit_df['Comment'].map(lambda t: clean(t, demojize=True)).to_numpy()
test_df = pd.read_csv('data/raw/binary_validation_inputs.csv')
X_test = test_df['Comment'].map(lambda t: clean(t, demojize=True)).to_numpy()
print('deduplicated OOF rows:', len(y))


## 2. Train the character-only SVM and save aligned probabilities

In [ ]:
def char_svm():
    features = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5), min_df=2, sublinear_tf=True)
    return make_pipeline(features, CalibratedClassifierCV(LinearSVC(C=0.5, class_weight='balanced'), cv=3))
CHAR_TAG = 'a_run21_svm_char_only'
char_dir = Path('artifacts/runs') / CHAR_TAG
char_dir.mkdir(parents=True, exist_ok=True)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
char_oof = np.zeros((len(y), 2))
for tr, va in skf.split(X, y):
    char_oof[va] = char_svm().fit(X[tr], y[tr]).predict_proba(X[va])
char_test = char_svm().fit(X, y).predict_proba(X_test)
np.save(char_dir / 'oof_probs.npy', char_oof)
np.save(char_dir / 'test_probs.npy', char_test)
print('char-only OOF:', f1_score(y, char_oof.argmax(1), average='macro'), accuracy_score(y, char_oof.argmax(1)))


## 3. Train the matched word+character SVM and MuRIL members

In [ ]:
SVM_TAG = 'a_run21_svm_wordchar'
MURIL_TAG = 'a_run21_muril_reinit2'
run([sys.executable, '-u', '-m', 'hastika.research.models.baseline_svm', '--task', 'a', '--tag', SVM_TAG, '--demojize'], log=f'artifacts/logs/{SVM_TAG}.log')
run([sys.executable, '-u', '-m', 'hastika.research.models.muril', '--tag', MURIL_TAG, '--model', 'google/muril-base-cased', '--folds', '5', '--seeds', '42', '--epochs', '6', '--bs', '8', '--grad-accum', '2', '--eval-bs', '32', '--select', 'last', '--reinit-layers', '2'], log=f'artifacts/logs/{MURIL_TAG}.log')
run([sys.executable, '-u', '-m', 'hastika.research.models.ensemble', '--task', 'a', '--tags', SVM_TAG, CHAR_TAG, MURIL_TAG, '--out', 'a_run21_ensemble'], log='artifacts/logs/a_run21_ensemble.log')
print('Run 21 three-member OOF and nested blend completed')


## 4. Save the individual comparison

In [ ]:
rows = []
for name, p in [('word+char SVM', np.load(Path('artifacts/runs') / SVM_TAG / 'oof_probs.npy')), ('char-only SVM', char_oof), ('MuRIL', np.load(Path('artifacts/runs') / MURIL_TAG / 'oof_probs.npy'))]:
    pred = p.argmax(1)
    rows.append({'model': name, 'macro_f1': f1_score(y, pred, average='macro'), 'accuracy': accuracy_score(y, pred)})
summary = pd.DataFrame(rows)
OUT = Path('/kaggle/working/task_a_run21_outputs')
OUT.mkdir(parents=True, exist_ok=True)
summary.to_csv(OUT / 'individual_oof_summary.csv', index=False)
print(summary.to_string(index=False))
print('The ensemble command above prints its in-sample search and nested estimate.')
print('DOWNLOAD RESULTS FROM:', OUT, 'and artifacts/runs')


## 5. Restore the canonical training file

In [ ]:
shutil.move(str(BACKUP_TRAIN), str(CANON_TRAIN))
print('restored', CANON_TRAIN)
